# 🌐 WebPulse — Google Colab Deployment (300 Parallel Workers)

Run **WebPulse** (Website Health, DNS, SSL Audit & Full-Page Screenshot platform) directly on Google Colab's high-speed cloud infrastructure with **300 concurrent workers**.

### Features in this Colab Runner:
- ⚡ **300 Parallel Workers** for high-throughput batch scanning (up to 1,000+ sites in minutes).
- 📸 Headless Chromium Playwright automation with Linux dependencies.
- 📂 Multi-column CSV & Excel (.xlsx, .xls) universal domain extraction.
- 📊 Batch PDF & multi-sheet Excel report exports by time window.
- 🔗 Instant public URL via Cloudflare Tunnel (100% free, no signup or tokens required).

### Step 1: Clone Repository and Install Dependencies

In [ ]:
# Clone the repository
!git clone https://github.com/AkashAP147/WEB_CHECKS.git
%cd WEB_CHECKS

# Install backend dependencies
!pip install -q -r backend/requirements.txt
!pip install -q python-multipart xlrd openpyxl reportlab uvicorn fastapi httpx dnspython cryptography certifi aiosqlite playwright pydantic-settings Pillow

# Install Playwright Chromium and Linux system dependencies
!playwright install chromium
!playwright install-deps chromium

print("✅ Setup complete!")

### Step 2: Launch WebPulse and Generate Public URL

In [ ]:
import subprocess
import time
import re

# Download Cloudflare Tunnel binary (free, no token needed)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Launch WebPulse Server (FastAPI + React bundle on port 8000)
print("Starting WebPulse backend on port 8000 (300 workers configured)...")
server_proc = subprocess.Popen(["python", "backend/run.py"])
time.sleep(3)

# Start Cloudflare Tunnel to expose port 8000 publicly
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Print the public web address
print("Connecting to Cloudflare Tunnel...")
for line in iter(tunnel_proc.stdout.readline, ''):
    if "trycloudflare.com" in line:
        url_match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if url_match:
            public_url = url_match.group(0)
            print("\n" + "=" * 65)
            print("🚀 WEBPULSE IS RUNNING!")
            print(f"👉 Open WebPulse: {public_url}")
            print("=" * 65 + "\n")
            break
